# Evaluation (TODO Latex figures without Type3 fonts!!!)

- Tables: Overall / By Label Class
- Figures: By Citation Count / By Year (difference)
- Samples (ACL only?)
- Confusion matrix?

```bash
# Compress all result files (exclude model weights)
tar -cvzf acl_docrel.tar.gz --exclude='*.bin' acl_docrel/*
```


In [ ]:
import re
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from collections import defaultdict
from experiments.utils import highlight_max
import matplotlib.patches as mpatches
from matplotlib.container import ErrorbarContainer

import os
import random
import json
import pickle
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook as tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from IPython.core.display import display

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
output_dir = './output/'
folds = 4

metric_cols = {
    'micro avg__precision': 'Precision (micro)',
    'micro avg__recall': 'Recall (micro)',
    'micro avg__f1-score': 'F1 (micro)',
    'macro avg__precision': 'Precision (macro)',
    'macro avg__recall': 'Recall (macro)',
    'macro avg__f1-score': 'F1 (macro)',
}

# Overall results

Table: System, P, R, F micro (std) F macro (std) |?for ACL-Anthology + CORD-19


In [ ]:
overall_rows = []
exp_names = []

for exp in os.listdir(output_dir):
    exp_dir = os.path.join(output_dir, exp)

    if not exp.endswith('_docrel') or os.path.isfile(exp_dir):
        continue
    
    print(f'Experiment: {exp}')
    
    for fold in range(1, folds+1):
        fold_dir = os.path.join(exp_dir, 'folds', str(fold))
        print(f'- Fold: {fold}')

        if not os.path.exists(fold_dir):
            logger.warning(f'Fold does not exist: {fold_dir}')
            continue

        for sys in os.listdir(fold_dir):
            sys_dir = os.path.join(fold_dir, sys)

            if not os.path.isdir(sys_dir) or sys.startswith('.') or sys.startswith('__'):
                continue

            try:
                metrics = json.load(open(os.path.join(sys_dir, 'metrics.json')))
                metrics = {k.replace('eval_', ''): v for k, v in metrics.items()}
                exp_name = exp.replace('acl_docrel', 'ACL Anthology').replace('cord19_docrel', 'CORD-19')
                row = {                    
                    'system': sys,
                    'experiment': exp_name,
                    'fold': fold,
                    'class': None,
                    'precision': 0.,
                    'recall': 0.,
                    'f1-score': 0.,
                    'support': 0,
                }
                cs = set([k.split('__')[0] for k in metrics.keys() if len(k.split('__')) == 2])

                for c in cs:
                    crow = row.copy()
                    crow['class'] = c
                    for m in ms:
                        crow[m] = metrics[c + '__' + m]
                    overall_rows.append(crow)
                    
                print(f'   - System: {sys}')

            except FileNotFoundError:
                logger.warning(f'Skip {sys_dir} (not all files exists, probably still running..)')
                pass
    #break

In [ ]:
df = pd.DataFrame(overall_rows)

In [ ]:
df

In [ ]:
systems = df['system'].unique().tolist()

In [ ]:
for sys in sorted(systems):
    print(sys)
    display(df[(df['system'] == sys) & (df['class'] == 'micro avg')])
    #display(df[(df['system'] == sys) & (df['class'] == 'macro avg')])
    
    

In [ ]:
df[(df['system'] == 'xlnet-base-cased') & (df['class'] == 'micro avg')]

In [ ]:
df.columns

In [ ]:
#agg_cols = {col_name:  ['mean', 'std', 'count'] for col_name, _ in metric_cols.items()}

agg_cols = {col_name:  ['mean', 'std',] for col_name in ['precision', 'recall', 'f1-score']}

overall_df = df[df['class'].isin(['micro avg', 'macro avg'])]\
    .groupby(['system','class','experiment', ])\
    .agg(agg_cols)\
    .unstack().unstack()\
    .swaplevel(0,2,axis=1).swaplevel(1,3,axis=1).sort_index(axis=1)
    

with pd.option_context('display.float_format', lambda x: ('%.3f' % x)[1:]):   
    display(
        overall_df.style.apply(highlight_max)
    )

In [ ]:
# pd.set_option('display.float_format', lambda x: '%.3f' % x)

with pd.option_context('display.float_format', lambda x: ('%.3f' % x)[1:]):   
    overall_tex = overall_df.to_latex().split(r'\midrule', 2)
    body_tex, footer_tex = overall_tex[1].split(r'\bottomrule', 2)    
    header_tex = overall_tex[0]
    
    
    # {} &       mean
    
    header_tex = re.sub(r'(.*)mean(.*)', '', header_tex)
    header_tex = re.sub(r'(.*)system(.*)', '', header_tex)
        
    #print(re.search(r'(.*)mean(.*)', header_tex))
    
    print(
        header_tex\
            .replace(r'multicolumn{2}{l}', r'multicolumn{1}{l}')\
            .replace(r'multicolumn{12}{l}', r'multicolumn{6}{l}')\
            .replace(r'multicolumn{6}{l}', r'multicolumn{3}{l}')\
            .replace('lrrrrrrrrrrrr', 'l|r|r|r|r|r|r')
    )
    print(r'\midrule')
    
    for i, line in enumerate(body_tex.split(r' \\')):
        if (i % 2) == 0:
            print(r'\rowcolor{Gray} ')
        
        print(
            line.replace('& .', r' $\pm$.')
        )
        print(r' \\')
        
    print(r'\bottomrule')
    print(footer_tex)

In [ ]:
odf.columns.levels

# By label class

- label X: sys_1_prec, sys_1_rec, sys_1_f1, sys_2_prec, ...

In [ ]:
selected_system = 'scibert-scivocab-uncased'
selected_systems = [
    'bert-base-cased',
    'scibert-scivocab-uncased'
]

label_df = df[df['system'].isin(selected_systems)]\
    .groupby(['class', 'experiment', 'system' ])\
    .agg({c: 'mean' for c in ['f1-score', 'precision', 'recall']})\
    .unstack()\
    .unstack()\
    .swaplevel(0,2,axis=1).sort_index(axis=1)\
    .reindex(sorted_label_classes)

    #.drop(columns=[('ACL Anthology', 'fold'), ('ACL Anthology', 'support')])\
    #.reindex(sorted_label_classes)
    
with pd.option_context('display.float_format', lambda x: ('%.3f' % x)[1:]):  
    display(label_df)

In [ ]:
label_classes = set([c.split('__')[0] for c in df.columns if '__' in c and ' avg' not in c])
label_classes

In [ ]:
sorted_label_classes = [
    'background',
    'conclusion',
    'discussion',
    'evaluation',
    'experiment',
    'introduction',    
    'method',
    'previous work',
    'related work',
    'results',
    'other',
    'none',
    'macro avg',
    'micro avg',
    'samples avg',
    'weighted avg'
]

#systems = df['system'].unique().tolist()
#footer_classes = ['other', 'none', 'macro avg', 'micro avg', ]


by_label_class_rows = []

for idx, row in df.iterrows():
    for col in row.keys():
        v = col.split('__')
        
        if len(v) == 2 and row['system'] in selected_systems:
            by_label_class_rows.append({
                'label': v[0],
                row['system'] + '__' + v[1]: row[col],
                #'fold': row['fold'],
            })

by_label_class_df = pd.DataFrame(by_label_class_rows)

In [ ]:
by_label_class_df.groupby('label').agg({selected_systems[0] + '__support': 'mean'}).values[:,0]

In [ ]:
by_label_class_df.groupby('label').agg(['mean', 'std', 'count']).reindex(sorted_label_classes)

# Samples

In [ ]:
sys_run_results_fp = os.path.join(exp_dir, 'folds', '1', 'bert-base-cased', 'results.csv')

results_cols = ['from_title', 'to_title', 'true', 'predicted']

results_df = pd.read_csv(sys_run_results_fp)
results_df.head()

In [ ]:
results_df[results_cols]

In [ ]:
without_predictions = pd.isnull(results_df['predicted']).sum()

print(f'Samples for that no probabilty was above classification threshold: {without_predictions:,} / {len(results_df):,}')

In [ ]:
results_df[results_df['predicted_background'] > 0][results_cols]

In [ ]:
results_df[results_df['true_background'] > 0][results_cols]

In [ ]:
results_df[results_df['predicted_related work'] > 0][results_cols]

In [ ]:
results_df[results_df['true_related work'] > 0][results_cols]

In [ ]:
results_df['true']

In [ ]:
results_df['true'].value_counts()[:20]

In [ ]:
results_df['predicted'].value_counts()

In [ ]:
results_df[results_df['predicted_related work'] > 0][results_cols]

# Multi-label confusion

```

missed classification for... |?predicted labels                                | true labels
                                introduction   | background |?experiment |?... |
introduction                 |        -        | 12         | 3          |?
(true_introduction=1,predicted_introductioin=0)


```

(color background min/max value)
